# Re-compressão do dataset 3W: Brotli → Snappy + upload S3

Clona o repositório [petrobras/3W](https://github.com/petrobras/3W) via Git LFS, re-comprime todos os parquets de Brotli para Snappy e faz upload para o S3.

> Pode rodar em qualquer runtime do Colab — GPU não é necessária.

## 1 — Instalar dependências

In [ ]:
!pip install -q boto3 pyarrow
!sudo apt-get install -y git-lfs -q
!git lfs install

## 2 — Clonar repositório 3W (Git LFS)

In [ ]:
!git clone https://github.com/petrobras/3W.git /content/3W
!cd /content/3W && git lfs pull

## 3 — Configuração

Preencha suas credenciais AWS e confirme o bucket de destino.

In [ ]:
import os

# Credenciais AWS (IAM key com s3:PutObject + s3:ListBucket no bucket abaixo)
os.environ["AWS_ACCESS_KEY_ID"]     = "SUA_ACCESS_KEY"
os.environ["AWS_SECRET_ACCESS_KEY"] = "SUA_SECRET_KEY"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

DATASET_LOCAL = "/content/3W/dataset"   # caminho local após git clone
BUCKET        = "streaming-3w"          # bucket S3 de destino
S3_PREFIX     = "3w/dataset"            # prefixo no S3
COMPRESSION   = "snappy"                # codec de destino (snappy ou zstd)
MAX_WORKERS   = 4                       # uploads paralelos

## 4 — Re-compressão e upload

In [ ]:
import io
import time
import pathlib
import boto3
import pyarrow.parquet as pq
import pyarrow as pa
from concurrent.futures import ThreadPoolExecutor, as_completed

s3 = boto3.client("s3")


def collect_parquet_files(local_dir: str) -> list:
    return sorted(pathlib.Path(local_dir).rglob("*.parquet"))


def normalize_schema(table: pa.Table) -> pa.Table:
    """
    Normaliza tipos problemáticos do dataset 3W para garantir compatibilidade com Spark:

    1. `class` (INT32 em alguns arquivos, DOUBLE em outros):
       Spark não consegue fazer coerção INT32 → DoubleType no leitor Parquet.
       Normaliza para float64 (DOUBLE) em todos os arquivos.

    2. `timestamp` (INT64 com anotação TIMESTAMP_NANOS):
       Mantido como int64 puro (sem anotação de timestamp) para evitar
       o erro "Illegal Parquet type: INT64 (TIMESTAMP(NANOS,false))" no Spark.
       pyarrow por padrão re-escreve timestamp[ns] com a anotação TIMESTAMP_NANOS;
       ao fazer cast para int64, o valor é preservado (nanossegundos desde epoch)
       sem a anotação problemática.
    """
    fields = table.schema
    columns = {}

    # Normaliza `class` para float64
    if "class" in fields.names:
        columns["class"] = table.column("class").cast(pa.float64())

    # Remove anotação TIMESTAMP_NANOS do índice temporal
    # pyarrow usa timestamp[ns] internamente; ao escrever com pa.int64()
    # grava como INT64 puro, sem anotação TIMESTAMP — lido nativamente pelo Spark.
    if "timestamp" in fields.names:
        ts_col = table.column("timestamp")
        if pa.types.is_timestamp(ts_col.type):
            # Converte timestamp[ns] → int64 preservando os nanosegundos
            columns["timestamp"] = ts_col.cast(pa.int64())

    if not columns:
        return table

    for col_name, new_col in columns.items():
        idx = table.schema.get_field_index(col_name)
        table = table.set_column(idx, col_name, new_col)

    return table


def recompress_and_upload(local_path, local_root, bucket, s3_prefix, compression):
    rel    = local_path.relative_to(local_root)
    s3_key = f"{s3_prefix}/{rel}".replace("\\", "/")
    result = {"path": str(local_path), "key": s3_key, "ok": False,
              "original_bytes": 0, "new_bytes": 0}
    try:
        result["original_bytes"] = local_path.stat().st_size
        table   = pq.read_table(str(local_path))   # suporta Brotli via libbrotli
        table   = normalize_schema(table)           # normaliza tipos para Spark
        buf_out = io.BytesIO()
        pq.write_table(table, buf_out, compression=compression)
        buf_out.seek(0)
        new_data = buf_out.read()
        result["new_bytes"] = len(new_data)
        s3.put_object(Bucket=bucket, Key=s3_key, Body=new_data)
        result["ok"] = True
    except Exception as e:
        result["error"] = str(e)
    return result


def run(local_dir, bucket, s3_prefix, compression, max_workers):
    root  = pathlib.Path(local_dir)
    files = collect_parquet_files(local_dir)
    total = len(files)
    print(f"Encontrados {total} arquivos em {local_dir}\n")

    t0 = time.time()
    ok = err = 0
    original_total = new_total = 0

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(recompress_and_upload, f, root, bucket, s3_prefix, compression): f
            for f in files
        }
        for i, future in enumerate(as_completed(futures), 1):
            r = future.result()
            if r["ok"]:
                ok += 1
                original_total += r["original_bytes"]
                new_total      += r["new_bytes"]
                status = "✓"
            else:
                err += 1
                status = f"✗ {r.get('error', '')[:60]}"

            if i % 50 == 0 or not r["ok"]:
                pct  = i / total * 100
                name = pathlib.Path(r["path"]).name
                print(f"  [{i:4d}/{total}] {pct:5.1f}%  {status}  {name}")

    elapsed = time.time() - t0
    ratio   = new_total / original_total if original_total else 1
    print(f"\n{'='*60}")
    print(f"  Concluído em {elapsed:.0f}s")
    print(f"  Sucesso : {ok}   Erros: {err}")
    print(f"  Original: {original_total/1e6:.1f} MB")
    print(f"  Novo    : {new_total/1e6:.1f} MB  ({ratio:.2f}x)")
    print(f"  S3      : s3://{bucket}/{s3_prefix}/")
    print(f"{'='*60}")


run(DATASET_LOCAL, BUCKET, S3_PREFIX, COMPRESSION, MAX_WORKERS)

## 5 — (Opcional) Verificar compressão de um arquivo no S3

In [ ]:
import boto3, io, pyarrow.parquet as pq

s3_check = boto3.client("s3")
resp = s3_check.get_object(
    Bucket=BUCKET,
    Key="3w/dataset/8/WELL-00019_20120601165020.parquet"
)
meta = pq.read_metadata(io.BytesIO(resp["Body"].read()))
print(meta.row_group(0).column(0).compression)  # deve imprimir SNAPPY